# 📊 VLM 视频筛选结果分析

分析 `batch_filter.py` 的筛选报告，查看结果并复制保留的视频。

In [1]:
from transformers import AutoProcessor
# 测试能不能加载
processor = AutoProcessor.from_pretrained("Qwen/Qwen3-VL-2B-Instruct")

/Users/I761836/Desktop/Semester 3/live-ultrasound-video-understanding/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [1]:
# Cell 1: 加载筛选报告
import json
import glob
from pathlib import Path

report_dir = "UltrasoundCrawler_KeyCode_20260323_v2/output/20260519_204257_youtube"
report_path = f"{report_dir}/vlm_filter_report.json"

import os
if os.path.exists(report_path):
    print(f"Loading: {report_path}")
    with open(report_path, 'r', encoding='utf-8') as f:
        report = json.load(f)
    videos = report['videos']
    print(f"Total: {report['total_videos']} videos")
    print(f"Time: {report['total_time_sec']/60:.1f} min")
    print(f"Summary: {report['summary']}")
else:
    print("No report found. Run: python batch_filter.py")
    report = None
    videos = []

No report found. Run: python batch_filter.py


In [ ]:
# Cell 2: Visualize
import matplotlib.pyplot as plt
import numpy as np

if videos:
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    
    # Pie chart
    dc = {}
    for v in videos:
        d = v['决策']
        dc[d] = dc.get(d, 0) + 1
    colors_map = {'保留':'#2ecc71','需要裁剪':'#f39c12','丢弃':'#e74c3c','错误':'#95a5a6'}
    axes[0].pie(dc.values(), labels=dc.keys(), colors=[colors_map.get(k,'#999') for k in dc.keys()], autopct='%1.0f%%')
    axes[0].set_title('Decision Distribution')
    
    # Bar chart by category
    cats = sorted(set(v.get('category','?') for v in videos))
    x = np.arange(len(cats))
    w = 0.25
    axes[1].bar(x-w, [sum(1 for v in videos if v.get('category')==c and v['决策']=='保留') for c in cats], w, label='Keep', color='#2ecc71')
    axes[1].bar(x, [sum(1 for v in videos if v.get('category')==c and v['决策']=='需要裁剪') for c in cats], w, label='Trim', color='#f39c12')
    axes[1].bar(x+w, [sum(1 for v in videos if v.get('category')==c and v['决策']=='丢弃') for c in cats], w, label='Discard', color='#e74c3c')
    axes[1].set_xticks(x)
    axes[1].set_xticklabels(cats, rotation=20, ha='right')
    axes[1].legend()
    axes[1].set_title('By Category')
    
    # Histogram
    scores = [v.get('质量评分', 0) for v in videos]
    axes[2].hist(scores, bins=20, color='#3498db', edgecolor='white')
    axes[2].axvline(x=70, color='green', linestyle='--', label='Keep threshold')
    axes[2].axvline(x=40, color='red', linestyle='--', label='Discard threshold')
    axes[2].legend()
    axes[2].set_title('Quality Score Distribution')
    
    plt.tight_layout()
    plt.show()

In [ ]:
# Cell 3: List kept videos
if videos:
    keep_videos = [v for v in videos if v['决策'] == '保留']
    trim_videos = [v for v in videos if v['决策'] == '需要裁剪']
    
    print(f"\n=== KEEP ({len(keep_videos)}) ===")
    for v in sorted(keep_videos, key=lambda x: x.get('质量评分',0), reverse=True):
        print(f"  [{v.get('质量评分',0):3d}] {v.get('category','?')}/{v['video_id']}")
        print(f"        {v.get('主要模态','?')} | {v.get('解剖部位',[])} | {v.get('判断理由','')}")
    
    print(f"\n=== TRIM ({len(trim_videos)}) ===")
    for v in sorted(trim_videos, key=lambda x: x.get('质量评分',0), reverse=True)[:10]:
        print(f"  [{v.get('质量评分',0):3d}] {v.get('category','?')}/{v['video_id']} | {v.get('判断理由','')}")
    if len(trim_videos) > 10:
        print(f"  ... and {len(trim_videos)-10} more")

In [ ]:
# Cell 4: Preview kept video frames
import cv2

def get_frames(path, n=5):
    cap = cv2.VideoCapture(str(path))
    if not cap.isOpened(): return []
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    indices = np.linspace(total*0.1, total*0.9, n, dtype=int)
    frames = []
    for idx in indices:
        cap.set(cv2.CAP_PROP_POS_FRAMES, int(idx))
        ret, f = cap.read()
        if ret: frames.append(cv2.cvtColor(f, cv2.COLOR_BGR2RGB))
    cap.release()
    return frames

if videos and keep_videos:
    n_show = min(5, len(keep_videos))
    fig, axes = plt.subplots(n_show, 5, figsize=(20, 4*n_show))
    if n_show == 1: axes = [axes]
    for row, v in enumerate(keep_videos[:n_show]):
        frames = get_frames(v['video_path'])
        for col, frame in enumerate(frames[:5]):
            axes[row][col].imshow(frame)
            axes[row][col].axis('off')
        axes[row][0].set_ylabel(f"{v['video_id'][:12]}\n[{v.get('质量评分',0)}]", rotation=0, labelpad=70, fontsize=8)
    plt.suptitle('Kept Videos Preview')
    plt.tight_layout()
    plt.show()

In [ ]:
# Cell 5: Copy kept videos to filtered_videos/
import shutil

OUTPUT_DIR = Path("filtered_videos")

if videos:
    keep_dir = OUTPUT_DIR / "keep"
    trim_dir = OUTPUT_DIR / "trim"
    keep_dir.mkdir(parents=True, exist_ok=True)
    trim_dir.mkdir(parents=True, exist_ok=True)
    
    print("Copying kept videos...")
    for v in keep_videos:
        src = Path(v['video_path'])
        if src.exists():
            dst = keep_dir / src.name
            if not dst.exists():
                shutil.copy2(src, dst)
                print(f"  + {src.name}")
    
    print(f"\nCopying trim videos...")
    for v in trim_videos:
        src = Path(v['video_path'])
        if src.exists():
            dst = trim_dir / src.name
            if not dst.exists():
                shutil.copy2(src, dst)
                print(f"  + {src.name}")
    
    print(f"\nDone! Keep: {keep_dir} | Trim: {trim_dir}")

In [ ]:
# Cell 6: Frame type and anatomy stats
if videos:
    all_types = {}
    for v in videos:
        for t, c in v.get('帧类型统计', {}).items():
            all_types[t] = all_types.get(t, 0) + c
    total_f = sum(all_types.values())
    print(f"Frame types (total {total_f} frames):")
    for t, c in sorted(all_types.items(), key=lambda x: x[1], reverse=True):
        print(f"  {t}: {c} ({c/total_f*100:.1f}%)")
    
    all_anat = {}
    for v in videos:
        for a in v.get('解剖部位', []):
            if a: all_anat[a] = all_anat.get(a, 0) + 1
    print(f"\nAnatomy detected:")
    for a, c in sorted(all_anat.items(), key=lambda x: x[1], reverse=True):
        print(f"  {a}: {c} videos")